In [ ]:
!pip install openai-whisper transformers sentencepiece torch nltk spacy yt-dlp pytube faster-whisper pydub langdetect gradio
!python -m spacy download en_core_web_sm

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')

import gradio as gr
import os
import nltk
import torch
import spacy
import re
import random
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, M2M100ForConditionalGeneration, M2M100Tokenizer
from faster_whisper import WhisperModel
import yt_dlp

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')
    nltk.download('stopwords')

nlp = spacy.load('en_core_web_sm')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print(f"🚀 AI Core initialized on {DEVICE}")

In [ ]:
class AI_Engine:
    def __init__(self):
        self.whisper = WhisperModel("medium", device=DEVICE, compute_type=COMPUTE_TYPE)

        summarizer_model_name = "facebook/bart-large-cnn"
        self.summarizer_tokenizer = AutoTokenizer.from_pretrained(summarizer_model_name)
        self.summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model_name).to(DEVICE)
        self.summarizer = self._perform_summarization

        self.trans_model_name = "facebook/m2m100_418M"
        self.trans_tokenizer = M2M100Tokenizer.from_pretrained(self.trans_model_name)
        self.trans_model = M2M100ForConditionalGeneration.from_pretrained(self.trans_model_name).to(DEVICE)

        qg_model_name = 'valhalla/t5-small-qa-qg-hl'
        self.qg_tokenizer = AutoTokenizer.from_pretrained(qg_model_name)
        self.qg_model = AutoModelForSeq2SeqLM.from_pretrained(qg_model_name).to(DEVICE)
        self.quiz_generator = self._perform_quiz_generation

    def _perform_summarization(self, text, max_length, min_length, do_sample):
        inputs = self.summarizer_tokenizer(
            text,
            max_length=1024,
            truncation=True,
            return_tensors="pt"
        ).to(DEVICE)

        summary_ids = self.summarizer_model.generate(
            inputs["input_ids"],
            num_beams=4,
            max_length=max_length,
            min_length=min_length,
            early_stopping=True,
            do_sample=do_sample
        )

        summary_text = self.summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return [{'summary_text': summary_text}]

    def _perform_quiz_generation(self, input_text):
        inputs = self.qg_tokenizer(input_text, return_tensors="pt", truncation=True).to(DEVICE)
        outputs = self.qg_model.generate(inputs["input_ids"], max_new_tokens=64)
        question_text = self.qg_tokenizer.decode(outputs[0], skip_special_tokens=True)
        return [{'generated_text': question_text}]

engine = AI_Engine()

In [ ]:
def download_video(url, progress=gr.Progress()):
    if not url: return None, "Please enter a URL."
    progress(0, desc="Initializing Download...")
    output_path = "downloads"
    os.makedirs(output_path, exist_ok=True)

    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': os.path.join(output_path, '%(title)s.%(ext)s'),
        'noplaylist': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            filename = ydl.prepare_filename(info)
            return filename, f"✅ Video downloaded: {info.get('title')}"
    except Exception as e:
        return None, f"❌ Error: {str(e)}"

def process_transcription(video_path, progress=gr.Progress()):
    if not video_path:
        return None, "Please download a video first."

    progress(0.2, desc="Loading Audio...")
    try:
        segments, info = engine.whisper.transcribe(video_path, beam_size=5)

        transcript_text = ""
        total_duration = info.duration

        for segment in segments:
            transcript_text += segment.text + " "
            if total_duration > 0:
                progress(min(segment.end / total_duration, 0.95), desc="Transcribing...")

        return transcript_text.strip(), "✅ Transcription Complete"
    except Exception as e:
        return None, f"❌ Error during transcription: {str(e)}"

In [ ]:
def generate_summary(text):
    if not text: return "No transcript available."

    chunks = [text[i:i+3000] for i in range(0, len(text), 3000)]
    summaries = []

    print("Generating Summary...")
    for chunk in chunks[:3]:
        try:
            input_len = len(chunk.split())
            max_l = min(150, int(input_len * 0.6))
            min_l = min(30, int(input_len * 0.2))

            if max_l > min_l:
                summary = engine.summarizer(chunk, max_length=max_l, min_length=min_l, do_sample=False)
                summaries.append(summary[0]['summary_text'])
        except Exception as e:
            print(f"Skipped chunk due to error: {e}")
            pass

    return " ".join(summaries)

def generate_notes(text):
    if not text: return "No transcript available."

    sentences = nltk.tokenize.sent_tokenize(text)
    if len(sentences) < 5:
        words = text.split()
        chunks = [' '.join(words[i:i+150]) for i in range(0, len(words), 150)]
    else:
        chunk_size = 7
        chunks = [' '.join(sentences[i:i+chunk_size]) for i in range(0, len(sentences), chunk_size)]

    print(f"DEBUG: Generated {len(chunks)} chunks. Processing top 5...")
    notes = []

    for i, chunk in enumerate(chunks[:5]):
        try:
            safe_chunk = chunk[:2500]
            res = engine.summarizer(safe_chunk, max_length=60, min_length=15, do_sample=False)
            point = res[0]['summary_text']
            notes.append(f"- {point}")
        except Exception as e:
            print(f"❌ Error extracting note from chunk {i}: {e}")

    if not notes:
        return "⚠️ Error: Could not generate notes. Check the 'Status' box or console for error details."

    return "### Key Takeaways:\n" + "\n".join(notes)

In [ ]:
def generate_quiz(text):
    if not text: return "No transcript available."

    print("Extracting entities for quiz...")
    doc = nlp(text[:5000])

    entities = [ent.text for ent in doc.ents if ent.label_ in ['DATE', 'ORG', 'PERSON', 'GPE', 'EVENT']]
    unique_entities = list(set(entities))

    if len(unique_entities) < 3:
        print("Not enough named entities, switching to Noun extraction...")
        unique_entities = [token.text for token in doc if token.pos_ == "NOUN" and len(token.text) > 4]
        unique_entities = list(set(unique_entities))

    random.shuffle(unique_entities)
    selected_answers = unique_entities[:5]

    quiz_output = ""

    for ans in selected_answers:
        for sent in nltk.tokenize.sent_tokenize(text):
            if ans in sent and len(sent) < 200:
                pattern = re.compile(re.escape(ans), re.IGNORECASE)
                input_text = pattern.sub(f"<hl>{ans}<hl>", sent)
                input_text = f"generate question: {input_text}"

                try:
                    q = engine.quiz_generator(input_text)[0]['generated_text']
                    quiz_output += f"**Q:** {q}\n**A:** ||{ans}||\n\n"
                    break
                except Exception as e:
                    print(f"Error generating quiz question for '{ans}': {e}")
                    continue

    if not quiz_output:
        return "Could not generate a quiz from this text. The content might be too abstract."

    return quiz_output

def translate_text(text, target_lang):
    if not text: return "No transcript."
    if not target_lang: return "Select a language."

    lang_map = {"Hindi": "hi", "French": "fr", "English": "en"}
    code = lang_map.get(target_lang, "en")

    print(f"Translating to {code}...")

    tokenizer = engine.trans_tokenizer
    model = engine.trans_model

    tokenizer.src_lang = "en"
    encoded = tokenizer(text[:600], return_tensors="pt").to(DEVICE)

    generated_tokens = model.generate(**encoded, forced_bos_token_id=tokenizer.get_lang_id(code))
    translated = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return translated

In [ ]:
custom_css = """
.gradio-container {
    background: linear-gradient(to bottom right, #0F172A, #1E1B4B) !important;
}

body, .prose, .markdown-text, label, span, p, h1, h2, h3, h4, h5, h6 {
    color: #F8FAFC !important;
    font-family: 'Inter', sans-serif;
}

textarea, input, .gr-box, .prose, #output_box, #status_box {
    background-color: rgba(30, 41, 59, 0.8) !important;
    color: #FFFFFF !important;
    border: 1px solid #334155 !important;
    border-radius: 12px !important;
    backdrop-filter: blur(5px);
}

h1 {
    background: linear-gradient(90deg, #818CF8, #22D3EE);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    text-align: center;
    font-weight: 800 !important;
    margin-bottom: 10px !important;
}

button.primary {
    background: linear-gradient(90deg, #4F46E5, #7C3AED) !important;
    color: white !important;
    border: none !important;
    font-weight: bold;
    transition: transform 0.1s;
}
button.primary:hover {
    transform: scale(1.02);
    box-shadow: 0 0 15px rgba(124, 58, 237, 0.5);
}

button.secondary {
    background-color: #334155 !important;
    color: #E2E8F0 !important;
    border: 1px solid #475569 !important;
}
button.secondary:hover {
    background-color: #475569 !important;
    color: white !important;
}

#video_player {
    border: 2px solid #6366F1;
    border-radius: 12px;
    background-color: #000;
    box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.5);
}

#status_box textarea {
    font-family: 'Courier New', monospace;
    color: #22D3EE !important;
    background-color: #020617 !important;
}

.tab-nav {
    border-bottom: 1px solid #475569 !important;
}
.selected {
    border-bottom: 3px solid #818CF8 !important;
    color: #818CF8 !important;
    background: transparent !important;
}

h3 {
    text-align: center;
    color: #94A3B8 !important;
}
"""

In [ ]:
with gr.Blocks(title="Study Mitra", theme=gr.themes.Soft(), css=custom_css) as demo:

    video_path_state = gr.State()
    transcript_state = gr.State()

    with gr.Row():
        with gr.Column():
            gr.Markdown("# Study Mitra")

    with gr.Row():
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown('### 1. Source & Status')

            url_input = gr.Textbox(label="YouTube URL", placeholder="Paste video link...", lines=1)
            download_btn = gr.Button("⬇️ Load Video", variant="primary", size="lg")

            status_msg = gr.Textbox(
                label="System Logs",
                value="System Ready...",
                interactive=False,
                lines=10,
                elem_id="status_box"
            )

        with gr.Column(scale=1):
            gr.Markdown('### 2. Video Preview')
            video_player = gr.Video(label="Player", interactive=True, elem_id="video_player")

    gr.Markdown("---")
    gr.Markdown('### 3. Intelligence Dashboard')

    with gr.Row(variant="panel"):
        with gr.Column():
            analyze_btn = gr.Button("✨ Transcribe & Analyze", variant="primary", size="lg", interactive=False)

            with gr.Tabs():

                with gr.TabItem("📄 Transcript"):
                    transcript_output = gr.Textbox(label="Full Text", lines=15, show_copy_button=True, elem_id="output_box")

                with gr.TabItem("📝 Summary"):
                    summ_btn = gr.Button("Generate Summary", variant="secondary")
                    summary_output = gr.Textbox(label="Abstract", lines=10, show_copy_button=True, elem_id="output_box")

                with gr.TabItem("📌 Notes"):
                    notes_btn = gr.Button("Extract Key Points", variant="secondary")
                    notes_output = gr.Markdown(elem_id="output_box")

                with gr.TabItem("❓ Quiz"):
                    quiz_btn = gr.Button("Generate Quiz", variant="secondary")
                    quiz_output = gr.Markdown(elem_id="output_box")

                with gr.TabItem("🌍 Translate"):
                    with gr.Row():
                        lang_select = gr.Dropdown(["Hindi", "French", "English"], label="Target Language", value="Hindi")
                        trans_btn = gr.Button("Translate", variant="secondary")
                    trans_output = gr.Textbox(label="Output", lines=10, elem_id="output_box")

    def on_download(url):
        path, msg = download_video(url)
        return path, path, msg, gr.update(interactive=(path is not None))

    download_btn.click(on_download, inputs=[url_input], outputs=[video_player, video_path_state, status_msg, analyze_btn])

    def on_transcribe(video_path):
        text, msg = process_transcription(video_path)
        return text, text, msg

    analyze_btn.click(on_transcribe, inputs=[video_path_state], outputs=[transcript_output, transcript_state, status_msg])

    summ_btn.click(generate_summary, inputs=[transcript_state], outputs=[summary_output])
    notes_btn.click(generate_notes, inputs=[transcript_state], outputs=[notes_output])
    quiz_btn.click(generate_quiz, inputs=[transcript_state], outputs=[quiz_output])
    trans_btn.click(translate_text, inputs=[transcript_state, lang_select], outputs=[trans_output])

demo.queue().launch(share=True, debug=True)